In [18]:
import pandas as pd


In [20]:
learn_df = pd.read_csv("modal_experiment_v1_training_norming_candidates_with_comprehension.csv"
                       ).drop(columns=["index", "index.1"])

# throw out wh-question and individual scenarios that performed worst during norming
to_keep = ["T1","T3","T5","T7","T9","T12"]
learn_df = learn_df[learn_df["item_id"].isin(to_keep)].reset_index(drop=True)

learn_df.head()


,phase,item_id,domain,training_flavor,syntax,D_status,E_status,temporal_orientation,order_pair_id,order_1,...,whole_scenario_response_options,expected_context_D,expected_context_E,expected_whole_scenario_deontic_interpretation,expected_whole_scenario_epistemic_interpretation,notes,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct
0,Training,T1,Museum entrance,Root/deontic,Declarative,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,Yes | No,D+,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Positive-only root training candidate. The con...,Why is Lina visiting the history museum?,For a class project,For a family trip,A
1,Training,T1,Museum entrance,Epistemic,Declarative,D unspecified,E+ (weak / indirect evidence),past perfect / present progressive or stative,TE,N-E,...,Yes | No,D should be 'The story does not say',E+,Expected non-deontic interpretation,Expected epistemic interpretation,Positive-only epistemic training candidate. We...,Why did Lina visit the history museum?,For a class project,For a family trip,A
2,Training,T3,Community center,Root/deontic,Declarative,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,Yes | No,D+,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Positive-only root training candidate. The con...,Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,A
3,Training,T3,Community center,Epistemic,Declarative,D unspecified,E+ (weak / indirect evidence),past perfect / present progressive or stative,TE,N-E,...,Yes | No,D should be 'The story does not say',E+,Expected non-deontic interpretation,Expected epistemic interpretation,Positive-only epistemic training candidate. We...,Which area has the front desk ruled out?,The fitness area,The swimming area,A
4,Training,T5,Parking,Root/deontic,Yes/no inversion,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,Yes | No,D+,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Positive-only root training candidate. The con...,What kind of event is Omar attending?,A museum exhibition,A concert,B


In [21]:
print(learn_df["E_status"].value_counts())

print(learn_df["D_status"].value_counts())


E_status
E unspecified / not ruled out    6
E+ (weak / indirect evidence)    6
Name: count, dtype: int64
D_status
D+               6
D unspecified    6
Name: count, dtype: int64


In [22]:
learn_df["D_status"] = learn_df["D_status"].map({"D+": 1, "D unspecified": 0, "D-": -1})
learn_df["E_status"] = learn_df["E_status"].map({"E+ (weak / indirect evidence)": 1, "E unspecified / not ruled out": 0})

learn_df["E_status"].tolist()


[0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]

In [23]:
order1_df = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_1",
            "context_order_1",
            "functional_auxiliary_target",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "order_1":"order",
                "context_order_1":"context_order"
                })
order1_df["order_group"] = (order1_df.reset_index().index % 4) < 2

order2_df = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_2",
            "context_order_2",
            "functional_auxiliary_target",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"]].rename(columns={
                "order_2":"order",
                "context_order_2":"context_order"
                })
order2_df["order_group"] = (order1_df.reset_index().index % 4) >= 2
learn_df = pd.concat([order1_df, order2_df])
learn_df["order_group"] = learn_df["order_group"].apply(int).apply(str)

print(order1_df["order_group"].tolist())
print(order2_df["order_group"].tolist())

print(learn_df["order"].tolist())
learn_df.groupby(["order_group", "training_flavor"])["order"].value_counts()


[True, True, False, False, True, True, False, False, True, True, False, False]
[False, False, True, True, False, False, True, True, False, False, True, True]
['N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N']


order_group  training_flavor  order
0            Epistemic        E-N      3
                              N-E      3
             Root/deontic     D-N      3
                              N-D      3
1            Epistemic        E-N      3
                              N-E      3
             Root/deontic     D-N      3
                              N-D      3
Name: count, dtype: int64

In [24]:

functional = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_group",
            "order",
            "context_order",
            "functional_auxiliary_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "functional_auxiliary_target":"alien_target_response",
                })
functional["modal_type"] = "functional"

lexical = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_group",
            "order",
            "context_order",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "lexical_adjective_target":"alien_target_response",
                })
lexical["modal_type"] = "lexical"

learn_df = pd.concat([functional, lexical])
learn_df.head()


,phase,item_id,domain,training_flavor,D_status,E_status,temporal_orientation,order_group,order,context_order,alien_target_response,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,modal_type
0,Training,T1,Museum entrance,Root/deontic,1,0,future eventive,1,N-D,Lina is visiting a history museum tomorrow for...,"Oh, so lina dax use the side entrance tomorrow.",Why is Lina visiting the history museum?,For a class project,For a family trip,A,functional
1,Training,T1,Museum entrance,Epistemic,0,1,past perfect / present progressive or stative,1,N-E,Lina visited a history museum this morning for...,"Oh, so lina dax have used the side entrance.",Why did Lina visit the history museum?,For a class project,For a family trip,A,functional
2,Training,T3,Community center,Root/deontic,1,0,future eventive,0,N-D,Nora is going to the community center this aft...,"Oh, so nora dax use the pool this afternoon.",Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,A,functional
3,Training,T3,Community center,Epistemic,0,1,past perfect / present progressive or stative,0,N-E,Nora is at the community center right now. The...,"Oh, so nora dax be swimming now.",Which area has the front desk ruled out?,The fitness area,The swimming area,A,functional
4,Training,T5,Parking,Root/deontic,1,0,future eventive,1,N-D,Omar is driving to a concert tomorrow. Omar's ...,"Oh, so dax Omar park in the west lot tomorrow?...",What kind of event is Omar attending?,A museum exhibition,A concert,B,functional


In [25]:
learn_df["comprehension_correct"] = learn_df.apply(
    lambda x: x["comprehension_option_A"] if x["comprehension_correct"] == "A" else x["comprehension_option_B"],
    axis=1
)

learn_df["question_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.split(" — ")[1] if " — " in x else ""
)
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.split(" — ")[0]
)

learn_df["group"] = learn_df.apply(
    lambda x: "D->E_" + x["modal_type"] if x["training_flavor"] == "Root/deontic" else "E->D_" + x["modal_type"],
    axis=1
) + "_" + learn_df["order_group"]


learn_df.to_csv("exp1_learning_stimulus.csv")
learn_df.head()


,phase,item_id,domain,training_flavor,D_status,E_status,temporal_orientation,order_group,order,context_order,alien_target_response,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,modal_type,question_response,group
0,Training,T1,Museum entrance,Root/deontic,1,0,future eventive,1,N-D,Lina is visiting a history museum tomorrow for...,"Oh, so lina dax use the side entrance tomorrow.",Why is Lina visiting the history museum?,For a class project,For a family trip,For a class project,functional,,D->E_functional_1
1,Training,T1,Museum entrance,Epistemic,0,1,past perfect / present progressive or stative,1,N-E,Lina visited a history museum this morning for...,"Oh, so lina dax have used the side entrance.",Why did Lina visit the history museum?,For a class project,For a family trip,For a class project,functional,,E->D_functional_1
2,Training,T3,Community center,Root/deontic,1,0,future eventive,0,N-D,Nora is going to the community center this aft...,"Oh, so nora dax use the pool this afternoon.",Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,The swimming pool,functional,,D->E_functional_0
3,Training,T3,Community center,Epistemic,0,1,past perfect / present progressive or stative,0,N-E,Nora is at the community center right now. The...,"Oh, so nora dax be swimming now.",Which area has the front desk ruled out?,The fitness area,The swimming area,The fitness area,functional,,E->D_functional_0
4,Training,T5,Parking,Root/deontic,1,0,future eventive,1,N-D,Omar is driving to a concert tomorrow. Omar's ...,"Oh, so dax Omar park in the west lot tomorrow?",What kind of event is Omar attending?,A museum exhibition,A concert,A concert,functional,Yes.,D->E_functional_1


In [26]:
learn_df.groupby(["group", "training_flavor", "modal_type"])["order"].value_counts()


group              training_flavor  modal_type  order
D->E_functional_0  Root/deontic     functional  D-N      3
                                                N-D      3
D->E_functional_1  Root/deontic     functional  D-N      3
                                                N-D      3
D->E_lexical_0     Root/deontic     lexical     D-N      3
                                                N-D      3
D->E_lexical_1     Root/deontic     lexical     D-N      3
                                                N-D      3
E->D_functional_0  Epistemic        functional  E-N      3
                                                N-E      3
E->D_functional_1  Epistemic        functional  E-N      3
                                                N-E      3
E->D_lexical_0     Epistemic        lexical     E-N      3
                                                N-E      3
E->D_lexical_1     Epistemic        lexical     E-N      3
                                                N-E      3
Na

In [27]:
from pathlib import Path

# Testing condition is between participants. Learning cells above are unchanged.
# Put these source CSVs beside the notebook (or in upload/).
def find_test_source(*filenames):
    for folder in (Path("."), Path("upload"), Path("../upload")):
        for filename in filenames:
            path = folder / filename
            if path.is_file():
                return path
    raise FileNotFoundError(f"Cannot find any of {filenames}")

condition_files = {
    "flavour_positive": find_test_source(
        "modal_experiment_v1_positive_flavour_stimulus_candidates.csv",
        "modal_experiment_v1_positive_flavour_stimulus_candidates(1).csv"),
    "flavour_negative": find_test_source(
        "modal_experiment_v1_negative_flavour_stimulus_candidates.csv"),
    "force_negative": find_test_source(
        "modal_experiment_v1_negative_force_stimulus_candidates.csv"),
}

test_sources = []
for condition, path in condition_files.items():
    source = pd.read_csv(path)
    source = source[source["phase"].isin(["Filler", "Critical Test"])].copy()
    source["condition"] = condition
    test_sources.append(source)
test_df = pd.concat(test_sources, ignore_index=True)
test_df.groupby(["condition", "phase"]).size()


condition         phase        
flavour_negative  Critical Test    16
                  Filler            8
flavour_positive  Critical Test    16
                  Filler            8
force_negative    Critical Test    16
                  Filler            8
dtype: int64

In [28]:
# Existing exclusions retained; revise these after the new norming results.
to_throw = ["F2", "F5", "C2", "C5"]
test_df = test_df[~test_df["item_id"].isin(to_throw)].reset_index(drop=True)
test_df.head()


,index,phase,item_id,domain,direction_or_condition,D_status,E_status,deontic_support_type,epistemic_support_type,temporal_orientation,...,context_only_possibility_question,context_only_necessity_question,whole_scenario_possibility_reading,whole_scenario_necessity_reading,context_only_possibility_response_options,context_only_necessity_response_options,expected_context_P,expected_context_Nec,expected_whole_scenario_possibility_interpretation,expected_whole_scenario_necessity_interpretation
0,30,Critical Test,C1,Zoo animal-care room,Root→Epistemic,D-,E+,explicit institutional restriction,open because relevant evidence is incomplete; ...,past perfect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,31,Critical Test,C1,Zoo animal-care room,Epistemic→Root,D+,E-,explicit credential-based permission,categorically ruled out by an explicit future ...,future eventive,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,34,Critical Test,C3,Student film studio,Root→Epistemic,D-,E+,explicit role-based restriction,open because the production record is incomplete,past perfect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,35,Critical Test,C3,Student film studio,Epistemic→Root,D+,E-,explicit badge-based permission,categorically ruled out by an explicit future ...,future eventive,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,36,Critical Test,C4,Hotel rooftop lounge,Root→Epistemic,D-,E+,explicit room-category restriction,open because Jordan's current location is unknown,present stative,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# `direction` keeps its original role: it identifies the LEARNING group.
# Flavour tests transfer to the other flavour; force tests stay in the learned
# flavour and test possibility -> necessity under negation.
def learning_direction(row):
    label = row["direction_or_condition"]
    if row["condition"] == "force_negative":
        if label.startswith("Root possibility"):
            return "D->E"  # Root-trained participants: root force test.
        if label.startswith("Epistemic possibility"):
            return "E->D"  # Epistemic-trained participants: epistemic force test.
    else:
        if label.startswith("Root→Epistemic"):
            return "D->E"
        if label.startswith("Epistemic→Root"):
            return "E->D"
    raise ValueError(f"Unrecognized critical condition: {label}")

criticals = test_df[test_df["phase"] == "Critical Test"].copy()
criticals["direction"] = criticals.apply(learning_direction, axis=1)

# Preserve the original logic: all fillers are shared by both learning groups,
# within their testing condition only.
fillers = test_df[test_df["phase"] == "Filler"].copy()
test_df = pd.concat([
    criticals,
    fillers.assign(direction="D->E"),
    fillers.assign(direction="E->D"),
], ignore_index=True)
test_df.groupby(["condition", "phase", "direction"]).size()


condition         phase          direction
flavour_negative  Critical Test  D->E         6
                                 E->D         6
                  Filler         D->E         6
                                 E->D         6
flavour_positive  Critical Test  D->E         6
                                 E->D         6
                  Filler         D->E         6
                                 E->D         6
force_negative    Critical Test  D->E         6
                                 E->D         6
                  Filler         D->E         6
                                 E->D         6
dtype: int64

In [30]:
# Keep the two order groups used by the learning phase. Each item appears
# once per group, and gets the opposite existing context version in group 1.
# Balance within each phase/pair first, then across the whole test block.
from itertools import product
from collections import Counter

# The current sources have one N-first pair per condition.
order_pair_orders = {
    "flavour_positive": {"N_FIRST": {"N-E-D", "N-D-E"}},
    "flavour_negative": {"N_FIRST": {"N-E-D", "N-D-E"}},
    "force_negative": {"N_FIRST": {"N-P-Nec", "N-Nec-P"}},
}
base_test = test_df.reset_index(drop=True).copy()
assert not base_test.duplicated(["condition", "direction", "item_id"]).any()
for row in base_test.itertuples(index=False):
    assert {row.order_1, row.order_2} == order_pair_orders[row.condition][row.order_pair_id]

order_groups = []
for (condition, direction), block in base_test.groupby(["condition", "direction"], sort=True):
    all_test_orders = sorted(set().union(*order_pair_orders[condition].values()))
    block = block.sort_values(["phase", "item_id"]).reset_index(drop=True)
    strata = list(block.groupby(["phase", "order_pair_id"]).indices.values())
    records = block.to_dict("records")
    best_score, best_choices = None, None
    # With the current exclusions: 12 items per block (4096 candidates).
    # This uses the actual
    # available orders rather than assuming the row index encodes the pair.
    for choices in product([0, 1], repeat=len(block)):
        if any(abs(sum(choices[i] for i in indices) * 2 - len(indices)) > 1
               for indices in strata):
            continue
        counts_by_group = []
        for group_code in [0, 1]:
            counts = Counter(
                row[f"order_{1 + (choices[i] ^ group_code)}"]
                for i, row in enumerate(records)
            )
            counts_by_group.append(counts)
        # Minimize the worst group's spread, then squared order counts.
        # Finally prefer an even split of first/second versions per phase.
        spread = max(max(c.get(o, 0) for o in all_test_orders) -
                     min(c.get(o, 0) for o in all_test_orders)
                     for c in counts_by_group)
        squared_counts = sum(c.get(o, 0) ** 2
                             for c in counts_by_group for o in all_test_orders)
        phase_split = sum(
            (sum(choices[i] for i in indices) * 2 - len(indices)) ** 2
            for indices in block.groupby("phase").indices.values()
        )
        score = (spread, squared_counts, phase_split)
        if best_score is None or score < best_score:
            best_score, best_choices = score, choices
    assert best_choices is not None, "No balanced assignment found"

    for group_code in [0, 1]:
        selected = block.copy()
        selected["order_group"] = str(group_code)
        selected["order_version"] = [1 + (choice ^ group_code) for choice in best_choices]
        selected["order"] = [row[f"order_{version}"] for row, version in
                             zip(records, selected["order_version"])]
        selected["context_order"] = [row[f"context_order_{version}"] for row, version in
                                     zip(records, selected["order_version"])]
        order_groups.append(selected)

test_df = pd.concat(order_groups, ignore_index=True)
assert test_df.groupby(["condition", "direction", "item_id", "order_group"]).size().eq(1).all()
assert test_df.groupby(["condition", "direction", "item_id"])["order_version"].nunique().eq(2).all()

# Inspect available pairs and the resulting counts for each condition.
pair_counts = base_test.groupby(["condition", "direction", "phase", "order_pair_id"]).size()
print("Available pair counts:\n", pair_counts)
test_df.groupby(["condition", "order_group", "direction", "phase"])["order"].value_counts()


Available pair counts:
 condition         direction  phase          order_pair_id
flavour_negative  D->E       Critical Test  N_FIRST          6
                             Filler         N_FIRST          6
                  E->D       Critical Test  N_FIRST          6
                             Filler         N_FIRST          6
flavour_positive  D->E       Critical Test  N_FIRST          6
                             Filler         N_FIRST          6
                  E->D       Critical Test  N_FIRST          6
                             Filler         N_FIRST          6
force_negative    D->E       Critical Test  N_FIRST          6
                             Filler         N_FIRST          6
                  E->D       Critical Test  N_FIRST          6
                             Filler         N_FIRST          6
dtype: int64


condition         order_group  direction  phase          order  
flavour_negative  0            D->E       Critical Test  N-D-E      3
                                                         N-E-D      3
                                          Filler         N-D-E      3
                                                         N-E-D      3
                               E->D       Critical Test  N-D-E      3
                                                         N-E-D      3
                                          Filler         N-D-E      3
                                                         N-E-D      3
                  1            D->E       Critical Test  N-D-E      3
                                                         N-E-D      3
                                          Filler         N-D-E      3
                                                         N-E-D      3
                               E->D       Critical Test  N-D-E      3
                         

In [32]:
# One testing condition per group, with a matching unchanged learning group.
assert test_df.groupby(["group", "item_id"]).size().eq(1).all()
assert test_df.groupby("group")["condition"].nunique().eq(1).all()
assert test_df.groupby("group")["learning_group"].nunique().eq(1).all()
assert set(test_df["learning_group"]) == set(learn_df["group"])
assert test_df.groupby("learning_group")["condition"].nunique().eq(3).all()
assert test_df["context_order"].notna().all()
assert test_df["alien_target_response"].notna().all()

# Within every condition, phase, and pair, order counts differ by at most one.
for (group, condition, phase), block in test_df.groupby(["group", "condition", "phase"]):
    for pair, orders in order_pair_orders[condition].items():
        paired = block[block["order"].isin(orders)]
        counts = paired["order"].value_counts().reindex(sorted(orders), fill_value=0)
        assert counts.max() - counts.min() <= 1

test_df.groupby(["condition", "group", "direction", "modal_type"])["order"].value_counts()


condition         group                               direction  modal_type  order  
flavour_negative  D->E_functional_0_flavour_negative  D->E       functional  N-D-E      6
                                                                             N-E-D      6
                  D->E_functional_1_flavour_negative  D->E       functional  N-D-E      6
                                                                             N-E-D      6
                  D->E_lexical_0_flavour_negative     D->E       lexical     N-D-E      6
                                                                             N-E-D      6
                  D->E_lexical_1_flavour_negative     D->E       lexical     N-D-E      6
                                                                             N-E-D      6
                  E->D_functional_0_flavour_negative  E->D       functional  N-D-E      6
                                                                             N-E-D      6
               

In [33]:
test_df[test_df["phase"] == "Critical Test"].groupby(
    ["condition", "group", "direction", "modal_type"])["order"].value_counts()


condition         group                               direction  modal_type  order  
flavour_negative  D->E_functional_0_flavour_negative  D->E       functional  N-D-E      3
                                                                             N-E-D      3
                  D->E_functional_1_flavour_negative  D->E       functional  N-D-E      3
                                                                             N-E-D      3
                  D->E_lexical_0_flavour_negative     D->E       lexical     N-D-E      3
                                                                             N-E-D      3
                  D->E_lexical_1_flavour_negative     D->E       lexical     N-D-E      3
                                                                             N-E-D      3
                  E->D_functional_0_flavour_negative  E->D       functional  N-D-E      3
                                                                             N-E-D      3
               

In [34]:
test_df[test_df["phase"] == "Filler"].groupby(
    ["condition", "group", "direction", "modal_type"])["order"].value_counts()


condition         group                               direction  modal_type  order  
flavour_negative  D->E_functional_0_flavour_negative  D->E       functional  N-D-E      3
                                                                             N-E-D      3
                  D->E_functional_1_flavour_negative  D->E       functional  N-D-E      3
                                                                             N-E-D      3
                  D->E_lexical_0_flavour_negative     D->E       lexical     N-D-E      3
                                                                             N-E-D      3
                  D->E_lexical_1_flavour_negative     D->E       lexical     N-D-E      3
                                                                             N-E-D      3
                  E->D_functional_0_flavour_negative  E->D       functional  N-D-E      3
                                                                             N-E-D      3
               

In [ ]:
test_df.head()


,phase,item_id,domain,condition,direction,direction_or_condition,D_status,E_status,P_status,Nec_status,temporal_orientation,order_group,order,context_order,expected_choice,alien_target_response,modal_type,learning_group,group
0,Critical Test,C1,Zoo animal-care room,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Aisha is volunteering at the zoo tomorrow. Ais...,Accept,"Oh, so it is not the case that Aisha dax work ...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
1,Critical Test,C3,Student film studio,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Grace is returning to the film studio tomorrow...,Accept,"Oh, so it is not the case that Grace dax use t...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
2,Critical Test,C4,Hotel rooftop lounge,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Jordan is staying at the hotel tomorrow. Jorda...,Accept,"Oh, so it is not the case that Jordan dax sit ...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
3,Critical Test,C6,Airport premium lounge,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-D-E,Marcus is flying from the airport again tomorr...,Accept,"Oh, so it is not the case that Marcus dax wait...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
4,Critical Test,C7,College radio station,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-D-E,Elena starts an internship at the radio statio...,Accept,"Oh, so it is not the case that Elena dax use t...",functional,D->E_functional_0,D->E_functional_0_flavour_negative


In [37]:
test_df.to_csv("exp1_test_stimulus.csv")


Participant assignment: choose one `group` from `exp1_test_stimulus.csv` and show only test rows matching it. Use that group's `learning_group` to select rows from the unchanged `exp1_learning_stimulus.csv`. For example, `D->E_functional_0_force_negative` uses learning group `D->E_functional_0`. There are 24 testing groups: 3 conditions × 2 learned flavours × 2 modal types × 2 order groups. The legacy `direction` label identifies the learning group; in force-negative testing, the flavour remains the learned flavour.

The existing `to_keep` and `to_throw` lists are retained for later revision. Testing uses only Critical Test and Filler rows from the three supplied sets. Fillers remain shared across the two learning directions, as in the original code. No learning cell source has changed. The separate training CSV referenced in cell 1 is still required to run the full notebook.


`expected_choice` is `Accept` or `Reject` for the displayed target, accounting for outer negation. For criticals this assumes successful transfer (to the other flavour, or from possibility to necessity); all current criticals predict `Accept`. Filler readings agree and give the same expected choice with or without transfer. Labels describe endorsement, not button positions or numeric response codes. Expected choices are derived from the status columns, not item IDs.